# needs hand correction about left and right!

# Hirriririir — Asian Dataset Evaluation (Thigh, Water)

Computes per-muscle Dice / Hausdorff metrics for Hirriririir SegResNetDS segmentations on the **MRI_data_asian** thigh water images.

- **Predictions**: `asian_segs_water/{subject}/Thigh/Water_thigh_seg.nii.gz` (integer labels 1–11)
- **Ground truth**: `MRI_data_asian/MRI_data/{subject}/Thigh/mask_muscles.nii.gz` (labels 1–13)

**Note**: The Hirriririir model does NOT distinguish L from R — both sides share the same integer label.  
**The Asian GT is unilateral**, so the model prediction covers both sides but the GT only one.  
This inflates false positives on the unlabeled side; results should be interpreted accordingly.  
Muscles absent from the model (`adductor_brevis`, `adductor_longus`, `gluteus_maximus`) → all-zero prediction.

In [ ]:
import glob
import os
import numpy as np
import pandas as pd
import SimpleITK as sitk
from dissector.evaluation import binary_cross_entropy, boundary_iou_3d, inter_slice_dice

In [ ]:
BOUNDARY_DISTANCE = 1

SEG_DIR    = os.path.join('..', 'asian_segs_water')
DATA_ROOT  = os.path.join('..', '..', 'MRI_data_asian', 'MRI_data')
RESULT_DIR = 'results_asian_water'
os.makedirs(RESULT_DIR, exist_ok=True)

# Hirriririir integer label map (no L/R distinction)
# 1=Sartorius, 2=Rectus_Femoris, 3=Vastus_Lateralis, 4=Vastus_Intermedius,
# 5=Vastus_Medialis, 6=Adductor_Magnus, 7=Gracilis, 8=Biceps_Femoris_Long,
# 9=Semitendinosus, 10=Semimembranosus, 11=Biceps_Femoris_Short

# (muscle_name, asian_gt_label, hirr_labels)
# hirr_labels: list of Hirriririir integer labels to OR, or None if not in model.
# No L/R split — both sides predicted under same label vs unilateral GT.
MUSCLES = [
    ('rectus_femoris',     1,  [2]),
    ('vastus_lateralis',   2,  [3]),
    ('vastus_intermedius', 3,  [4]),
    ('vastus_medialis',    4,  [5]),
    ('sartorius',          5,  [1]),
    ('gracilis',           6,  [7]),
    ('biceps_femoris',     7,  [8, 11]),  # long + short head merged
    ('semitendinosus',     8,  [9]),
    ('semimembranosus',    9,  [10]),
    ('adductor_brevis',    10, None),
    ('adductor_longus',    11, None),
    ('adductor_magnus',    12, [6]),
    ('gluteus_maximus',    13, None),
]

seg_files = sorted(glob.glob(os.path.join(SEG_DIR, '*', 'Thigh', 'Water_thigh_seg.nii.gz')))
print(f'SEG_DIR   : {os.path.abspath(SEG_DIR)}')
print(f'DATA_ROOT : {os.path.abspath(DATA_ROOT)}')
print(f'RESULT_DIR: {os.path.abspath(RESULT_DIR)}')
print(f'Found     : {len(seg_files)} segmentation files')

In [ ]:
def evaluate_muscle(muscle_name, gt_label, hirr_labels, seg_files, result_dir):
    """
    hirr_labels: list of integer labels to OR together, or None (→ all-zero pred).
    """
    results = []
    for seg_path in seg_files:
        parts   = seg_path.replace('\\', '/').split('/')
        subject = parts[-3]

        gt_path = os.path.join(DATA_ROOT, subject, 'Thigh', 'mask_muscles.nii.gz')
        if not os.path.exists(gt_path):
            print(f'  GT not found: {gt_path}, skipping')
            continue

        gt_image = sitk.ReadImage(gt_path)
        gt       = sitk.Cast(gt_image == gt_label, sitk.sitkUInt8)
        gt_arr   = sitk.GetArrayFromImage(gt).astype(float)

        seg_raw  = sitk.GetArrayFromImage(sitk.ReadImage(seg_path))
        if hirr_labels is not None:
            pred_arr = np.zeros_like(seg_raw, dtype=np.uint8)
            for lbl in hirr_labels:
                pred_arr |= (seg_raw == lbl).astype(np.uint8)
        else:
            pred_arr = np.zeros_like(seg_raw, dtype=np.uint8)

        pred_sitk = sitk.GetImageFromArray(pred_arr)
        pred_sitk.CopyInformation(gt_image)
        pred = sitk.Cast(pred_sitk, sitk.sitkUInt8)

        dice_filter = sitk.LabelOverlapMeasuresImageFilter()
        dice_filter.Execute(gt, pred)

        if gt_arr.sum() > 0 and pred_arr.sum() > 0:
            hd_filter = sitk.HausdorffDistanceImageFilter()
            hd_filter.Execute(gt, pred)
            hd = hd_filter.GetHausdorffDistance()
        else:
            hd = np.nan

        results.append({
            'subject':                              subject,
            'seg_file':                             os.path.basename(seg_path),
            f'{muscle_name}_dice':                  dice_filter.GetDiceCoefficient(),
            f'{muscle_name}_hausdorff':             hd,
            f'{muscle_name}_jaccard':               dice_filter.GetJaccardCoefficient(),
            f'{muscle_name}_volume_similarity':     dice_filter.GetVolumeSimilarity(),
            f'{muscle_name}_false_negative':        dice_filter.GetFalseNegativeError(),
            f'{muscle_name}_false_positive':        dice_filter.GetFalsePositiveError(),
            f'{muscle_name}_bce':                   binary_cross_entropy(gt_arr, pred_arr.astype(float)),
            f'{muscle_name}_boundary_iou_3d':       boundary_iou_3d(BOUNDARY_DISTANCE, gt_arr, pred_arr.astype(float)),
            f'{muscle_name}_inter_slice_dice_pred': inter_slice_dice(pred_arr.astype(float)),
            f'{muscle_name}_inter_slice_dice_gt':   inter_slice_dice(gt_arr),
        })

    df       = pd.DataFrame(results)
    csv_path = os.path.join(result_dir, f'df_{muscle_name}_hirriririir_asian_water.csv')
    df.to_csv(csv_path, index=False)
    print(f'  Saved {len(df)} rows → {csv_path}')
    return df

In [ ]:
# ── Run evaluation ────────────────────────────────────────────────────────────

dfs = {}
for muscle_name, gt_label, hirr_labels in MUSCLES:
    print(f'\n── {muscle_name}  (GT={gt_label}, labels={hirr_labels}) ──')
    dfs[muscle_name] = evaluate_muscle(
        muscle_name, gt_label, hirr_labels, seg_files, RESULT_DIR
    )

print('\nDone.')

In [ ]:
# ── Summary ───────────────────────────────────────────────────────────────────

from IPython.display import display

summary_rows = []
for muscle_name, df in dfs.items():
    dice_col = f'{muscle_name}_dice'
    hd_col   = f'{muscle_name}_hausdorff'
    summary_rows.append({
        'muscle':         muscle_name,
        'n':              len(df),
        'dice_mean':      df[dice_col].mean(),
        'dice_std':       df[dice_col].std(),
        'hausdorff_mean': df[hd_col].mean(),
        'hausdorff_std':  df[hd_col].std(),
    })

summary = pd.DataFrame(summary_rows).set_index('muscle')
summary_path = os.path.join(RESULT_DIR, 'summary_hirriririir_asian_water.csv')
summary.to_csv(summary_path)
print(f'Summary saved → {summary_path}\n')
display(summary.round(4))